# Round 2 - Notebook D: retrain stage 2 on a correctly reconstructed W&I+LOCNESS set

The earlier notebooks read **A, B, C and the combined ABC** W&I training files, so every sentence appeared twice and 94% of
the 5% holdout also occurred in training. This notebook re-runs **only stage 2** (about 15 min of training per model) on
A+B+C (34,308 pairs) with a holdout of 5% of the unique source sentences, reusing the stage-1 models of Notebook C,
then evaluates exactly as before and writes a new blind BEA-2019 test file.

**Setup (Kaggle):** GPU **T4 x2** - Internet **on** - **Add Input -> Your Work -> `R2_C_retrain_core`**:
* the version that ran `JOBS = ["final"]` (its output has `t5v3_model/` and `clean_test.parquet`), and
* the version that ran `JOBS = ["nogate"]` (its output has `model_nogate/`).

If Kaggle lets you attach only one version at a time, run this notebook twice (once with each); the notebook runs
whichever job's model it finds. No C4 dataset is needed. ~2.5 h for `final`, ~1.5 h for `nogate`.

Send back **`r2D_results.zip`**, and upload **`bea19_test_submission_dedup.zip`** to CodaBench as before.

In [ ]:
# ============================== CONFIG ==============================
import os; os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
JOBS = ["final", "nogate"]       # a job runs only if its stage-1 model is found among the inputs
QUICK_TEST = False               # True: ~25-min smoke test on small subsets
SESSION_BUDGET_H = 11.6
RESULTS_NAME, EXPORT_NAME = "r2D_results.json", "r2D_results.zip"
SEED = 42
TEST_N, C4_EVAL_N, SUBSET_SEED = 25_000, (5_000 if not QUICK_TEST else 300), 2026   # same C4 subset as notebooks A-C
PER_DEVICE_BATCH, GRAD_ACCUM, EPOCHS = 16, 4, 1
LEARNING_RATE, WARMUP_FRAC, WEIGHT_DECAY = 3e-4, 0.06, 0.01
LABEL_SMOOTHING, SRC_DROPOUT_P = 0.1, 0.05
EVAL_SAVE_STEPS = 2500
STAGE2_EPOCHS, STAGE2_LR, STAGE2_GA = 3, 1e-4, 2         # stage-2 global batch 64 (unchanged recipe)
HOLDOUT_FRAC = 0.05                                      # of the unique W&I source sentences
GATE_INIT_BIAS = 2.0
TAUS = [0.0, 0.02, 0.05, 0.1, 0.15, 0.2, 0.3, 0.5]
MAX_TRAIN_EVAL_RATIO = 2.0
RUN2_CLEAN_TEST_SHA256 = "6f724bdbb7d1321ed3011158d06ca49cb0672291bdd9c4d258c3a26ca0e3440d"


In [ ]:
# ---------------------------------------------------------------- installs
import subprocess, sys, importlib
def pip(*a):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *a], capture_output=True, text=True)
    if r.returncode: print("pip warning:", " ".join(a), r.stderr[-300:])
pip("errant==3.0.2", "sentencepiece", "pyarrow")
import spacy
try:
    spacy.load("en_core_web_sm")
except OSError:
    v = ".".join(spacy.__version__.split(".")[:2]) + ".0"
    pip(f"https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-{v}/en_core_web_sm-{v}-py3-none-any.whl")
spacy.load("en_core_web_sm")
print("spaCy", spacy.__version__, "| en_core_web_sm OK")


In [ ]:
# ---------------------------------------------------------------- imports + run bookkeeping
import os, re, io, csv, json, gc, math, time, random, shutil, tarfile, zipfile, urllib.request, contextlib
from pathlib import Path
from collections import Counter
from difflib import SequenceMatcher
import numpy as np, pandas as pd, pyarrow as pa, pyarrow.parquet as pq
import torch
from torch.utils.data import Dataset
from transformers import (AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM,
                          DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments)
os.environ["WANDB_DISABLED"] = "true"
OUT = Path("/kaggle/working"); EV = OUT / "eval_data"; EV.mkdir(exist_ok=True)
T0 = time.time()
def hours_left(): return SESSION_BUDGET_H - (time.time() - T0) / 3600
def budget_ok(need_h, what=""):
    ok = hours_left() > need_h
    if not ok: print(f"!! budget guard: {hours_left():.2f} h left < {need_h} h needed -> skipping {what}")
    return ok
RES_PATH = OUT / RESULTS_NAME
RESULTS = json.loads(RES_PATH.read_text()) if RES_PATH.exists() else {}
def save_results(): RES_PATH.write_text(json.dumps(RESULTS, indent=1, default=str))
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device={DEV} | GPUs={torch.cuda.device_count()} | budget={SESSION_BUDGET_H} h | QUICK_TEST={QUICK_TEST}")


In [ ]:
# ---------------------------------------------------------------- official evaluation data + scorers
CONLL_URL = "https://www.comp.nus.edu.sg/~nlp/conll14st/conll14st-test-data.tar.gz"
M2_URL    = "https://www.comp.nus.edu.sg/~nlp/sw/m2scorer.tar.gz"
BEA_URL   = "https://www.cl.cam.ac.uk/research/nl/bea2019st/data/wi+locness_v2.1.bea19.tar.gz"
JF_BASE   = "https://raw.githubusercontent.com/keisks/jfleg/master/"

def fetch(url, dest):
    if not Path(dest).exists(): urllib.request.urlretrieve(url, dest)
    return Path(dest)

def untar(tgz, into):
    with tarfile.open(tgz) as t: t.extractall(into)

# CoNLL-2014 test (official, 1,312 sentences, 2 annotators)
if not (EV / "conll14st-test-data").exists(): untar(fetch(CONLL_URL, EV / "conll.tgz"), EV)
CONLL_M2 = EV / "conll14st-test-data" / "noalt" / "official-2014.combined.m2"
# official MaxMatch scorer, ported to Python 3 OFFLINE and embedded: Kaggle now runs Python 3.13, which removed
# lib2to3, so converting it at run time is no longer possible. (Port verified: reproduces 0.4852 / 0.3096 exactly.)
import base64 as _b64, gzip as _gz
with tarfile.open(fileobj=io.BytesIO(_gz.decompress(_b64.b64decode("...")))) as _t:
    _t.extractall(EV)                                   # always overwrite with the ported copy
M2_SCRIPT = EV / "m2scorer" / "scripts" / "m2scorer.py"
# W&I+LOCNESS v2.1 (BEA-2019): dev gold M2, blind test input, train M2
if not (EV / "wi+locness").exists(): untar(fetch(BEA_URL, EV / "wi.tgz"), EV)
BEA_DEV_M2 = EV / "wi+locness" / "m2" / "ABCN.dev.gold.bea19.m2"
BEA_TEST_ORIG = EV / "wi+locness" / "test" / "ABCN.test.bea19.orig"
# JFLEG test (747 sentences, 4 references) + the OFFICIAL GLEU script (Napoles et al., 2016 version)
JF = EV / "jfleg"; JF.mkdir(exist_ok=True)
for n in ["test/test.src"] + [f"test/test.ref{k}" for k in range(4)] + ["eval/gleu.py"]:
    fetch(JF_BASE + n, JF / Path(n).name)
sys.path.insert(0, str(JF))
from gleu import GLEU

def m2_sources(path): return [l[2:].strip() for l in open(path, encoding="utf-8") if l.startswith("S ")]
def read_m2(path):
    out = []
    for block in Path(path).read_text(encoding="utf-8").strip().split("\n\n"):
        lines = block.split("\n"); src = lines[0][2:].strip(); ed = {}
        for ln in lines[1:]:
            if not ln.startswith("A "): continue
            f = ln[2:].split("|||"); s, e = map(int, f[0].split()); ann = int(f[-1])
            ed.setdefault(ann, [])
            if f[1] != "noop": ed[ann].append((s, e, f[2]))
        out.append((src, ed))
    return out
def apply_m2_edits(src, edits):
    toks = src.split(); off = 0
    for s, e, cor in sorted(edits, key=lambda x: (x[0], x[1])):
        c = cor.split() if cor and cor != "-NONE-" else []
        toks[s + off:e + off] = c; off += len(c) - (e - s)
    return " ".join(toks)

CONLL_SRC = m2_sources(CONLL_M2)
if QUICK_TEST:                       # smoke test: first 200 CoNLL sentences with the matching gold
    _b = Path(CONLL_M2).read_text(encoding="utf-8").strip().split("\n\n")[:200]
    CONLL_M2 = EV / "conll_quick.m2"; CONLL_M2.write_text("\n\n".join(_b) + "\n\n", encoding="utf-8")
    CONLL_SRC = m2_sources(CONLL_M2)
BEA_DEV = read_m2(BEA_DEV_M2)
BEA_SRC = [s for s, _ in BEA_DEV]
BEA_TGT = [apply_m2_edits(s, ed.get(0, [])) for s, ed in BEA_DEV]      # first annotator (as in the paper)
BEA_TEST_SRC = [l.rstrip("\n") for l in open(BEA_TEST_ORIG, encoding="utf-8")]
JF_SRC = [l.rstrip("\n") for l in open(JF / "test.src", encoding="utf-8")]
JF_REFS = [JF / f"test.ref{k}" for k in range(4)]
if QUICK_TEST:
    BEA_N, JF_N = 200, 120
else:
    BEA_N, JF_N = len(BEA_SRC), len(JF_SRC)
print(f"CoNLL-2014 {len(CONLL_SRC)} | BEA dev {len(BEA_SRC)} | BEA test {len(BEA_TEST_SRC)} | JFLEG {len(JF_SRC)}")


In [ ]:
# ---------------------------------------------------------------- retokenizers + official scorers
# Both benchmarks ship PRE-TOKENIZED sources and both scorers compare whitespace tokens, so every
# system output is retokenized with the tokenizer that best reproduces the benchmark's own convention
# (round-trip test on the sources: regex 99.3% on CoNLL-2014, spaCy 98.7% on BEA-2019).
import spacy
_NLP = spacy.load("en_core_web_sm", disable=["parser", "ner", "lemmatizer", "tagger", "attribute_ruler"])
def spacy_tok(s): return " ".join(t.text for t in _NLP.make_doc(str(s)) if not t.is_space)
_CLITIC = re.compile(r"(?i)(\w)(n't|'s|'re|'ve|'ll|'d|'m)\b")
_PA = re.compile(r"(\S)([.,!?;:%)\]\"](?:\s|$))"); _PB = re.compile(r"(^|\s)([(\[\"])(\S)")
def regex_tok(s):
    s = _CLITIC.sub(r"\1 \2", str(s))
    for _ in range(3): s = _PA.sub(r"\1 \2", s)
    return re.sub(r"\s+", " ", _PB.sub(r"\1\2 \3", s)).strip()
RETOK = {"conll": regex_tok, "jfleg": regex_tok, "bea": spacy_tok}
def norm_ws(s): return re.sub(r"\s+", " ", str(s)).strip()

def official_m2(hyps, tag, gold=None):
    gold = gold or CONLL_M2
    n = sum(1 for l in open(gold, encoding="utf-8") if l.startswith("S "))
    assert len(hyps) == n, f"{tag}: {len(hyps)} hyps vs {n} gold"
    hp = OUT / f"_m2_{tag}.txt"; hp.write_text("\n".join(hyps) + "\n", encoding="utf-8")
    r = subprocess.run([sys.executable, str(M2_SCRIPT), str(hp), str(gold)], capture_output=True, text=True,
                       timeout=7200, env={**os.environ, "PYTHONUTF8": "1"})
    g = dict(re.findall(r"(Precision|Recall|F_0\.5)\s*:\s*([0-9.]+)", r.stdout))
    if not g: print("m2scorer failed:", r.stderr[-400:]); return {"error": r.stderr[-400:]}
    return {"P": float(g["Precision"]), "R": float(g["Recall"]), "F05": float(g["F_0.5"])}

import errant
from errant.commands import compare_m2
_ANN = {"a": None}
def ann():
    if _ANN["a"] is None: _ANN["a"] = errant.load("en")
    return _ANN["a"]
def errant_official(srcs, hyps, gold_m2, tag):
    """Hypothesis M2 via ERRANT (whitespace tokens, as errant_parallel), then the official errant_compare."""
    a = ann(); hp = OUT / f"_errant_{tag}.m2"
    with open(hp, "w", encoding="utf-8") as f:
        for s, h in zip(srcs, hyps):
            o = a.parse(s); ed = a.annotate(o, a.parse(h))
            f.write("S " + s + "\n")
            if not ed: f.write("A -1 -1|||noop|||-NONE-|||REQUIRED|||-NONE-|||0\n")
            for e in ed: f.write(e.to_m2() + "\n")
            f.write("\n")
    gold_use = gold_m2
    if len(srcs) < len(m2_sources(gold_m2)):                       # QUICK_TEST subsets
        blocks = Path(gold_m2).read_text(encoding="utf-8").strip().split("\n\n")[:len(srcs)]
        gold_use = OUT / f"_gold_{tag}.m2"; gold_use.write_text("\n\n".join(blocks) + "\n\n", encoding="utf-8")
    buf = io.StringIO(); old = sys.argv
    sys.argv = ["compare_m2", "-hyp", str(hp), "-ref", str(gold_use)]
    try:
        with contextlib.redirect_stdout(buf): compare_m2.main()
    finally: sys.argv = old
    found = re.findall(r"^(\d+)\s+(\d+)\s+(\d+)\s+([0-9.]+)\s+([0-9.]+)\s+([0-9.]+)\s*$", buf.getvalue(), re.M)
    if not found: print("errant_compare output not parsed"); return {"error": buf.getvalue()[-400:]}
    tp, fp, fn, p, r, f = found[-1]
    return {"TP": int(tp), "FP": int(fp), "FN": int(fn), "P": float(p), "R": float(r), "F05": float(f)}

def jfleg_gleu(hyps, n=None):
    """OFFICIAL JFLEG GLEU (keisks/jfleg eval/gleu.py, 500 random-reference iterations), source-aware."""
    n = n or len(hyps)
    sp, hp = OUT / "_jf_src.txt", OUT / "_jf_hyp.txt"
    sp.write_text("\n".join(JF_SRC[:n]) + "\n", encoding="utf-8"); hp.write_text("\n".join(hyps[:n]) + "\n", encoding="utf-8")
    refs = []
    for k, rp in enumerate(JF_REFS):
        p = OUT / f"_jf_ref{k}.txt"; p.write_text("\n".join(open(rp, encoding="utf-8").read().split("\n")[:n]) + "\n", encoding="utf-8"); refs.append(str(p))
    g = GLEU(4); g.load_sources(str(sp)); g.load_references(refs)
    mean, std, ci = list(g.run_iterations(num_iterations=500, source=str(sp), hypothesis=str(hp), per_sent=False))[0]
    return {"GLEU": float(mean), "std": float(std), "ci95": ci}

def lev(a, b):
    a, b = str(a).split(), str(b).split(); prev = list(range(len(b) + 1))
    for i, x in enumerate(a, 1):
        cur = [i]
        for j, y in enumerate(b, 1): cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (x != y)))
        prev = cur
    return prev[-1]
def outcome_table(srcs, tgts, preds):
    c = Counter(); n = len(srcs)
    for s, t, p in zip(srcs, tgts, preds):
        if p.strip() == t.strip(): c["perfect"] += 1
        elif p.strip() == s.strip(): c["unchanged"] += 1
        else:
            ds, dp = lev(s, t), lev(p, t)
            c["improved" if dp < ds else ("lateral" if dp == ds else "harm")] += 1
    o = {k: round(100 * c[k] / n, 2) for k in ("perfect", "improved", "unchanged", "lateral", "harm")}
    o["edited_pct"] = round(100 * sum(a.strip() != b.strip() for a, b in zip(srcs, preds)) / n, 2)
    return o
def single_ref_errant(srcs, tgts, preds):
    """Edit-level P/R/F0.5 against ONE reference (the in-domain C4 test), plus per-sentence counts."""
    a = ann(); tp = fp = fn = 0; per = []
    for s, t, p in zip(srcs, tgts, preds):
        o = a.parse(s)
        gold = set() if t == s else {(e.o_start, e.o_end, e.c_str) for e in a.annotate(o, a.parse(t)) if e.type != "noop"}
        hyp = set() if p == s else {(e.o_start, e.o_end, e.c_str) for e in a.annotate(o, a.parse(p)) if e.type != "noop"}
        x = (len(gold & hyp), len(hyp - gold), len(gold - hyp)); per.append(x)
        tp += x[0]; fp += x[1]; fn += x[2]
    P = tp / max(tp + fp, 1); R = tp / max(tp + fn, 1)
    return {"P": P, "R": R, "F05": (1.25 * P * R / (0.25 * P + R)) if (P + R) else 0.0, "TP": tp, "FP": fp, "FN": fn}, per
print("scorers ready")


In [ ]:
# ---------------------------------------------------------------- FLAN-T5 decoding, exactly as in the paper
PREFIX = "Correct the grammar in this sentence: "
MAX_INPUT_LEN = MAX_TARGET_LEN = 128
GEN_KWARGS = dict(max_length=128, num_beams=5, length_penalty=0.7, no_repeat_ngram_size=3,
                  early_stopping=True, min_length=1)
REFINE_MAX_EDITS = 3
GEN_BATCH = 64
def token_edit(a, b):
    sm = SequenceMatcher(None, str(a).split(), str(b).split())
    return sum(max(i2 - i1, j2 - j1) for tag, i1, i2, j1, j2 in sm.get_opcodes() if tag != "equal")
def load_t5(path):
    tok = AutoTokenizer.from_pretrained(str(path)); m = AutoModelForSeq2SeqLM.from_pretrained(str(path)).to(DEV).eval()
    return tok, m
@torch.no_grad()
def t5_gen(tok, m, texts, gen=None, bs=GEN_BATCH, desc=""):
    kw = dict(GEN_KWARGS); kw.update(gen or {}); out = []
    for i in range(0, len(texts), bs):
        enc = tok([PREFIX + str(s) for s in texts[i:i + bs]], return_tensors="pt", padding=True,
                  truncation=True, max_length=MAX_INPUT_LEN).to(m.device)
        out += tok.batch_decode(m.generate(**enc, **kw), skip_special_tokens=True)
        if desc and (i // bs) % 25 == 0: print(f"   {desc} {i}/{len(texts)}", flush=True)
    return out
def refine(tok, m, srcs, pass1, gen=None):
    need = [i for i, p in enumerate(pass1) if p.strip() != str(srcs[i]).strip()]
    out = list(pass1)
    if need:
        r = t5_gen(tok, m, [pass1[k] for k in need], gen)
        for k, i in enumerate(need):
            if token_edit(pass1[i], r[k]) <= REFINE_MAX_EDITS: out[i] = r[k]
    return out
@torch.no_grad()
def mean_nll(tok, m, ins, outs, bs=GEN_BATCH):
    res = []
    for i in range(0, len(ins), bs):
        enc = tok([PREFIX + str(s) for s in ins[i:i + bs]], return_tensors="pt", padding=True, truncation=True,
                  max_length=MAX_INPUT_LEN).to(m.device)
        lab = tok(text_target=[str(t) for t in outs[i:i + bs]], return_tensors="pt", padding=True, truncation=True,
                  max_length=MAX_TARGET_LEN).to(m.device)
        labels = lab["input_ids"].masked_fill(lab["attention_mask"] == 0, -100)
        lp = torch.log_softmax(m(**enc, labels=labels).logits.float(), dim=-1)
        nll = -lp.gather(2, labels.clamp(min=0).unsqueeze(-1)).squeeze(-1); mask = (labels != -100).float()
        res += ((nll * mask).sum(1) / mask.sum(1).clamp(min=1)).tolist()
    return res
def margins(tok, m, srcs, preds):
    idx = [i for i, (s, p) in enumerate(zip(srcs, preds)) if str(p).strip() != str(s).strip()]
    mg = [None] * len(srcs)
    if idx:
        a = mean_nll(tok, m, [srcs[i] for i in idx], [preds[i] for i in idx])
        b = mean_nll(tok, m, [srcs[i] for i in idx], [srcs[i] for i in idx])
        for k, i in enumerate(idx): mg[i] = b[k] - a[k]
    return mg
def revert(srcs, preds, mg, tau):
    if tau is None: return list(preds)
    return [s if (x is not None and x < tau) else p for s, p, x in zip(srcs, preds, mg)]
def decode_variants(tok, m, srcs, lps=(0.7,), refines=(True,), taus=(0.0,), desc=""):
    """All (length-penalty x refinement x threshold) variants from ONE beam decode per length penalty."""
    out = {}
    for lp in lps:
        p1 = t5_gen(tok, m, srcs, {"length_penalty": lp}, desc=f"{desc} lp={lp}")
        for rf in refines:
            p = refine(tok, m, srcs, p1, {"length_penalty": lp}) if rf else list(p1)
            mg = margins(tok, m, srcs, p) if any(t is not None for t in taus) else None
            for tau in taus:
                out[(lp, rf, tau)] = [norm_ws(x) for x in revert(srcs, p, mg, tau)]
    return out
def vkey(lp, rf, tau): return f"lp{lp}_ref{int(rf)}_tau{'none' if tau is None else tau}"


In [ ]:
# ---------------------------------------------------------------- inputs: stage-1 models + the clean C4 test split
import hashlib
def find_dirs(name):
    hits = []
    for root, dirs, files in os.walk("/kaggle/input", followlinks=True):
        if Path(root).name == name and "config.json" in files and any(f.endswith((".safetensors", ".bin")) for f in files):
            hits.append(Path(root))
    return hits
S1 = {"final": find_dirs("t5v3_model"), "nogate": find_dirs("model_nogate")}
for job, hits in S1.items(): print(f"stage-1 model for '{job}':", [str(h) for h in hits] or "NOT FOUND")
S1 = {job: hits[0] for job, hits in S1.items() if hits}
JOBS = [j for j in JOBS if j in S1]
assert JOBS, ("No stage-1 model found. Add Input -> Your Work -> R2_C_retrain_core: the version that ran JOBS=['final'] "
              "(contains t5v3_model/) and/or the version that ran JOBS=['nogate'] (contains model_nogate/).")
te_path = None
for root, dirs, files in os.walk("/kaggle/input", followlinks=True):
    if "clean_test.parquet" in files: te_path = Path(root) / "clean_test.parquet"; break
assert te_path is not None, "clean_test.parquet not found in the attached Notebook C output"
te = pq.read_table(te_path).to_pandas()
sha = hashlib.sha256("\n".join(f"{s}\t{t}" for s, t in zip(te.src, te.tgt)).encode("utf-8")).hexdigest()
assert sha == RUN2_CLEAN_TEST_SHA256, "the attached clean test split differs from the paper's"
SUB_IDX = np.sort(np.random.default_rng(SUBSET_SEED).permutation(TEST_N)[:C4_EVAL_N])
C4_SRC = [te.src[i] for i in SUB_IDX]; C4_TGT = [te.tgt[i] for i in SUB_IDX]; del te
print("jobs to run:", JOBS, "| clean C4 test split verified (sha256) | C4 subset", len(C4_SRC))

# ---------------------------------------------------------------- W&I+LOCNESS stage-2 data, reconstructed CORRECTLY
wi_files = [p for p in sorted((EV / "wi+locness" / "m2").glob("*.train.gold.bea19.m2")) if not p.name.startswith("ABC.")]
assert [p.name for p in wi_files] == ["A.train.gold.bea19.m2", "B.train.gold.bea19.m2", "C.train.gold.bea19.m2"], wi_files
pairs = [(s, apply_m2_edits(s, ed.get(0, []))) for p_ in wi_files for s, ed in read_m2(p_)]
assert len(pairs) == 34_308, len(pairs)
srcs_u = sorted(set(s for s, _ in pairs))
perm = np.random.default_rng(SEED).permutation(len(srcs_u)); n_hold = int(round(len(srcs_u) * HOLDOUT_FRAC))
hold_src = set(srcs_u[i] for i in perm[:n_hold])
tr_idx = [i for i, (s, _) in enumerate(pairs) if s not in hold_src]
ho_idx = [i for i, (s, _) in enumerate(pairs) if s in hold_src]
assert not (set(pairs[i][0] for i in tr_idx) & hold_src) and len(tr_idx) + len(ho_idx) == len(pairs)
if QUICK_TEST: tr_idx, ho_idx = tr_idx[:3000], ho_idx[:200]
pq.write_table(pa.table({"src": [pairs[i][0] for i in tr_idx], "tgt": [pairs[i][1] for i in tr_idx]}), OUT / "wi_train_dedup.parquet")
pq.write_table(pa.table({"src": [pairs[i][0] for i in ho_idx], "tgt": [pairs[i][1] for i in ho_idx]}), OUT / "wi_holdout_dedup.parquet")
HOLD_SRC, HOLD_TGT = [pairs[i][0] for i in ho_idx], [pairs[i][1] for i in ho_idx]
(OUT / "wi_holdout_index.txt").write_text("\n".join(map(str, ho_idx)) + "\n")      # indices into A+B+C file order
RESULTS["wi_split"] = {"files": [p.name for p in wi_files], "pairs": len(pairs), "unique_sources": len(srcs_u),
                       "unique_pairs": len(set(pairs)), "holdout_sources": n_hold, "train_pairs": len(tr_idx),
                       "holdout_pairs": len(ho_idx), "holdout_rule": "5% of unique source sentences, seed 42"}
save_results(); print("W&I+LOCNESS:", json.dumps(RESULTS["wi_split"]))


In [ ]:
# ---------------------------------------------------------------- the copy gate (the paper's module, two fixes)
# (1) num_items_in_batch is dropped before it reaches T5 (newer transformers inject it);
# (2) the model is loaded with EAGER attention -- the gate needs the cross-attention weights, which faster
#     attention backends do not return. Everything else is the June module verbatim.
GATE_SRC = """
import torch, torch.nn as nn, torch.nn.functional as F
from transformers import T5ForConditionalGeneration
from transformers.modeling_outputs import Seq2SeqLMOutput
class T5WithCopyGate(T5ForConditionalGeneration):
    is_copy_gate = True
    def __init__(self, config):
        super().__init__(config)
        self.copy_gate = nn.Linear(2 * config.d_model, 1)
        nn.init.zeros_(self.copy_gate.weight); nn.init.constant_(self.copy_gate.bias, 2.0)
        self._last_pgen_mean = None
    def forward(self, input_ids=None, attention_mask=None, decoder_input_ids=None, labels=None, copy_src_ids=None, **kwargs):
        kwargs.pop("num_items_in_batch", None)
        # generation passes use_cache / past_key_values; training and evaluation do not. Outside generation the
        # model returns ONLY loss + logits: multi-GPU DataParallel gathers every returned tensor onto GPU 0, and the
        # attentions / hidden states / cache of all layers exhausted its memory at the first step.
        gen_mode = bool(kwargs.get("use_cache")) or kwargs.get("past_key_values") is not None
        if not gen_mode: kwargs["use_cache"] = False
        if labels is not None and decoder_input_ids is None and kwargs.get("decoder_inputs_embeds") is None:
            decoder_input_ids = self._shift_right(labels)
        kwargs["output_attentions"] = True; kwargs["output_hidden_states"] = True; kwargs["return_dict"] = True
        out = super().forward(input_ids=input_ids, attention_mask=attention_mask, decoder_input_ids=decoder_input_ids,
                              labels=None, **kwargs)
        if copy_src_ids is None: copy_src_ids = input_ids
        if copy_src_ids is None:
            new_logits = out.logits
        else:
            assert out.cross_attentions is not None and out.cross_attentions[-1] is not None, \\
                "no cross-attention weights returned -- load the model with attn_implementation='eager'"
            attn = out.cross_attentions[-1].mean(dim=1).float()
            if attention_mask is not None: attn = attn * attention_mask.unsqueeze(1).float()
            enc_h = out.encoder_last_hidden_state.float(); dec_h = out.decoder_hidden_states[-1].float()
            context = torch.bmm(attn, enc_h)
            p_gen = torch.sigmoid(self.copy_gate(torch.cat([dec_h, context], dim=-1)).float())
            self._last_pgen_mean = float(p_gen.mean().detach())
            probs = torch.softmax(out.logits.float(), dim=-1) * p_gen
            idx = copy_src_ids.unsqueeze(1).expand(-1, probs.size(1), -1)
            probs.scatter_add_(2, idx, attn * (1.0 - p_gen))
            new_logits = torch.log(probs + 1e-9)
        loss = None
        if labels is not None:
            loss = F.cross_entropy(new_logits.view(-1, new_logits.size(-1)), labels.view(-1).to(new_logits.device), ignore_index=-100)
        if not gen_mode:
            return Seq2SeqLMOutput(loss=loss, logits=new_logits)
        return Seq2SeqLMOutput(loss=loss, logits=new_logits, past_key_values=out.past_key_values,
                               decoder_hidden_states=out.decoder_hidden_states, cross_attentions=out.cross_attentions,
                               encoder_last_hidden_state=out.encoder_last_hidden_state)
    def prepare_inputs_for_generation(self, input_ids, **kwargs):
        mi = super().prepare_inputs_for_generation(input_ids, **kwargs)
        if kwargs.get("copy_src_ids") is not None: mi["copy_src_ids"] = kwargs["copy_src_ids"]
        return mi
"""
(OUT / "t5_copy_gate.py").write_text(GATE_SRC)
sys.path.insert(0, str(OUT))
from t5_copy_gate import T5WithCopyGate

# gate-aware generation: replaces t5_gen from the decoding cell (refine / decode_variants call it by name)
@torch.no_grad()
def t5_gen(tok, m, texts, gen=None, bs=GEN_BATCH, desc=""):
    kw = dict(GEN_KWARGS); kw.update(gen or {}); out = []
    for i in range(0, len(texts), bs):
        enc = tok([PREFIX + str(s) for s in texts[i:i + bs]], return_tensors="pt", padding=True, truncation=True,
                  max_length=MAX_INPUT_LEN).to(m.device)
        extra = {"copy_src_ids": enc["input_ids"]} if getattr(m, "is_copy_gate", False) else {}
        out += tok.batch_decode(m.generate(**enc, **kw, **extra), skip_special_tokens=True)
        if desc and (i // bs) % 25 == 0: print(f"   {desc} {i}/{len(texts)}", flush=True)
    return out
def load_model(path, gate=False):
    tok = AutoTokenizer.from_pretrained(str(path))
    m = (T5WithCopyGate.from_pretrained(str(path), attn_implementation="eager") if gate
         else AutoModelForSeq2SeqLM.from_pretrained(str(path)))
    return tok, m.to(DEV).eval()


In [ ]:
# ---------------------------------------------------------------- training, with the loss-normalisation guard
from transformers import TrainerCallback
import inspect
def _drop(text, p, rng):
    w = str(text).split()
    if p <= 0 or len(w) < 4: return text
    k = [x for x in w if rng.random() > p]
    return " ".join(k) if k else text
class GECDataset(Dataset):
    def __init__(self, path, tok, dp=0.0, seed=42):
        t = pq.read_table(path); self.s, self.t, self.tok, self.dp, self.seed = t.column("src"), t.column("tgt"), tok, dp, seed
    def __len__(self): return len(self.s)
    def __getitem__(self, i):
        s, t = str(self.s[i].as_py()), str(self.t[i].as_py())
        if self.dp > 0 and s != t: s = _drop(s, self.dp, random.Random(self.seed * 1_000_003 + i))   # edit pairs only
        e = self.tok(PREFIX + s, max_length=MAX_INPUT_LEN, truncation=True)
        e["labels"] = self.tok(text_target=t, max_length=MAX_TARGET_LEN, truncation=True)["input_ids"]
        return e

class LossScaleGuard(TrainerCallback):
    """At the first evaluation, compare the recent training loss with the validation loss. A ratio near 4 means the
    loss is not divided by gradient accumulation (the June defect): stop at once instead of training for hours."""
    def __init__(self): self.train, self.ratio, self.gnorm = [], None, []
    def on_log(self, args, state, control, logs=None, **kw):
        logs = logs or {}
        if "loss" in logs: self.train.append((state.global_step, logs["loss"]))
        if "grad_norm" in logs: self.gnorm.append(logs["grad_norm"])
        if "eval_loss" in logs and self.ratio is None:
            recent = [l for s, l in self.train if s > state.global_step - max(1000, 4 * args.logging_steps)] or [l for _, l in self.train]
            if recent:
                self.ratio = float(np.mean(recent) / logs["eval_loss"])
                print(f"[guard] train/eval loss ratio at step {state.global_step}: {self.ratio:.2f} (correct ~1.1-1.3)", flush=True)
                if self.ratio > MAX_TRAIN_EVAL_RATIO:
                    print("!! LOSS-SCALING DEFECT: the training loss is not normalised by gradient accumulation. Stopping.")
                    control.should_training_stop = True
        return control

def train(tag, train_path, val_path, init, save_dir, gate=False, stage=1, seed=SEED):
    if (save_dir / "config.json").exists(): print("exists:", save_dir); return
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    tok = AutoTokenizer.from_pretrained(str(init))
    if gate:
        m = T5WithCopyGate.from_pretrained(str(init), attn_implementation="eager")
        m.copy_gate.weight.data.zero_(); m.copy_gate.bias.data.fill_(GATE_INIT_BIAS)     # as the paper
    else:
        m = AutoModelForSeq2SeqLM.from_pretrained(str(init))
    tr = GECDataset(train_path, tok, SRC_DROPOUT_P if stage == 1 else 0.0, seed)
    va = GECDataset(val_path, tok, 0.0, seed + 1)
    if QUICK_TEST and stage == 1: va = torch.utils.data.Subset(va, range(200))
    n_gpu = max(1, torch.cuda.device_count())
    # micro-batch x accumulation; the global batch (128 stage 1, 64 stage 2) is fixed. If a configuration runs out
    # of GPU memory (the copy gate keeps several vocabulary-sized fp32 tensors per token), halve the micro-batch and
    # double the accumulation -- the same global batch and the same recipe.
    ga0 = GRAD_ACCUM if stage == 1 else STAGE2_GA
    for pdb, ga in ((PER_DEVICE_BATCH, ga0), (PER_DEVICE_BATCH // 2, ga0 * 2), (PER_DEVICE_BATCH // 4, ga0 * 4)):
        try:
            return _train_once(tag, tok, m, tr, va, save_dir, gate, stage, seed, n_gpu, pdb, ga)
        except torch.OutOfMemoryError:
            pass
        # cleanup OUTSIDE the except block: while an exception is being handled its traceback keeps the failed
        # trainer (optimizer states, activations) alive on the GPU
        print(f"!! {tag}: out of GPU memory at micro-batch {pdb} x accumulation {ga} -- retrying with half the micro-batch")
        m = None; gc.collect(); torch.cuda.empty_cache(); shutil.rmtree(OUT / f"ck_{tag}", ignore_errors=True)
        m = (T5WithCopyGate.from_pretrained(str(init), attn_implementation="eager") if gate
             else AutoModelForSeq2SeqLM.from_pretrained(str(init)))
        if gate: m.copy_gate.weight.data.zero_(); m.copy_gate.bias.data.fill_(GATE_INIT_BIAS)
    raise RuntimeError(f"{tag}: out of memory even at micro-batch {PER_DEVICE_BATCH // 4}")

def _train_once(tag, tok, m, tr, va, save_dir, gate, stage, seed, n_gpu, pdb, ga):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    total = math.ceil(len(tr) / (pdb * ga * n_gpu)) * (EPOCHS if stage == 1 else STAGE2_EPOCHS)
    bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8     # T4 -> fp16, as June
    a = dict(output_dir=str(OUT / f"ck_{tag}"), learning_rate=LEARNING_RATE if stage == 1 else STAGE2_LR,
             lr_scheduler_type="cosine", per_device_train_batch_size=pdb, per_device_eval_batch_size=16 if gate else 64,
             gradient_accumulation_steps=ga, num_train_epochs=EPOCHS if stage == 1 else STAGE2_EPOCHS,
             warmup_steps=max(1, int(total * WARMUP_FRAC)), weight_decay=WEIGHT_DECAY, label_smoothing_factor=LABEL_SMOOTHING,
             logging_steps=(500 if stage == 1 else 50) if not QUICK_TEST else 10, save_total_limit=2 if stage == 1 else 1,
             predict_with_generate=False, fp16=torch.cuda.is_available() and not bf16, bf16=bf16, dataloader_num_workers=2,
             report_to=[], seed=seed, load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False)
    if stage == 1: a.update(save_strategy="steps", save_steps=EVAL_SAVE_STEPS, eval_steps=EVAL_SAVE_STEPS); ev = "steps"
    else: a.update(save_strategy="epoch"); ev = "epoch"
    a["eval_strategy" if "eval_strategy" in inspect.signature(Seq2SeqTrainingArguments.__init__).parameters else "evaluation_strategy"] = ev
    guard = LossScaleGuard()
    t = Seq2SeqTrainer(model=m, args=Seq2SeqTrainingArguments(**a), train_dataset=tr, eval_dataset=va,
                       data_collator=DataCollatorForSeq2Seq(tok, model=m, label_pad_token_id=-100), callbacks=[guard])
    t0 = time.time(); o = t.train(); dt = time.time() - t0
    rec = {"stage": stage, "gate": gate, "train_rows": len(tr), "steps": int(o.global_step), "planned_steps": total,
           "global_batch": pdb * ga * n_gpu, "micro_batch": pdb, "grad_accum": ga,
           "warmup_steps": max(1, int(total * WARMUP_FRAC)),
           "train_seconds": round(dt, 1), "examples_per_s": round(len(tr) * a["num_train_epochs"] / dt, 2),
           "train_loss": float(o.training_loss), "best_checkpoint": t.state.best_model_checkpoint,
           "best_eval_loss": t.state.best_metric, "train_eval_loss_ratio": guard.ratio,
           "median_grad_norm": float(np.median(guard.gnorm)) if guard.gnorm else None}
    if gate:
        m.eval(); b = DataCollatorForSeq2Seq(tok, model=m)([va[i] for i in range(min(32, len(va)))])
        with torch.no_grad(): m(**{k: v.to(m.device) for k, v in b.items()})
        rec["mean_p_gen_val_probe"] = m._last_pgen_mean
    RESULTS[f"train|{tag}"] = rec; save_results(); json.dump(t.state.log_history, open(OUT / f"trainer_log_{tag}.json", "w"))
    if guard.ratio is not None and guard.ratio > MAX_TRAIN_EVAL_RATIO:
        raise RuntimeError(f"{tag}: loss-scaling defect detected (ratio {guard.ratio:.2f}) -- send the log to Claude")
    t.save_model(str(save_dir)); tok.save_pretrained(str(save_dir))
    shutil.rmtree(OUT / f"ck_{tag}", ignore_errors=True); del t, m; gc.collect(); torch.cuda.empty_cache()
    print(tag, json.dumps(rec, default=str))


In [ ]:
# ---------------------------------------------------------------- evaluation
def edit_sets(srcs, tgts, preds):
    a = ann(); out = []
    for s, t, p in zip(srcs, tgts, preds):
        o = a.parse(s)
        g = set() if t == s else {(e.o_start, e.o_end, e.c_str) for e in a.annotate(o, a.parse(t)) if e.type != "noop"}
        h = set() if p == s else {(e.o_start, e.o_end, e.c_str) for e in a.annotate(o, a.parse(p)) if e.type != "noop"}
        out.append((g, h))
    return out
def tune_tau(tok, m, srcs, tgts):
    """Pick the revert threshold on HELD-OUT data (lp 0.7 + refinement), maximising edit-level F0.5 (as the paper)."""
    p = refine(tok, m, srcs, t5_gen(tok, m, srcs, desc="tune")); mg = margins(tok, m, srcs, p); es = edit_sets(srcs, tgts, p)
    best, curve = (None, -1.0), {}
    for tau in [None] + TAUS:
        tp = fp = fn = 0
        for (g, h), x in zip(es, mg):
            use = h if (tau is None or x is None or x >= tau) else set()
            tp += len(g & use); fp += len(use - g); fn += len(g - use)
        P_ = tp / max(tp + fp, 1); R_ = tp / max(tp + fn, 1); F_ = 1.25 * P_ * R_ / (0.25 * P_ + R_) if (P_ + R_) else 0.0
        curve[str(tau)] = round(F_, 4)
        if F_ > best[1]: best = (tau, F_)
    print("   tau curve:", curve, "-> selected", best[0]); return best[0], curve

def score_all(tag, tok, m, taus=(None,), refines=(True,), with_test=False, test_tau=None):
    """lp 0.7 throughout (the length-penalty sweep had no effect, notebook B). Saves every variant's predictions."""
    r = {}
    sets = (("c4", C4_SRC, None), ("bea_dev", BEA_SRC[:BEA_N], "bea"), ("conll", CONLL_SRC, "conll"), ("jfleg", JF_SRC[:JF_N], "jfleg"))
    for name, srcs, rt in sets:
        V = decode_variants(tok, m, srcs, (0.7,), refines, taus, desc=f"{tag}/{name}")
        cols = {}
        for (lp, rf, tau), p in V.items():
            k = f"ref{int(rf)}_tau{'none' if tau is None else tau}"
            if name == "c4":
                mm, _ = single_ref_errant(C4_SRC, C4_TGT, p); r[f"c4|{k}"] = {**mm, **outcome_table(C4_SRC, C4_TGT, p)}; cols[k] = p
            else:
                pr = [RETOK[rt](x) for x in p]; cols[k] = pr
                r[f"{name}|{k}"] = (errant_official(srcs, pr, BEA_DEV_M2, f"{tag}_{name}_{k}") if name == "bea_dev" else
                                    official_m2(pr, f"{tag}_{name}_{k}") if name == "conll" else jfleg_gleu(pr, JF_N))
        base = {"c4": {"test_index": SUB_IDX, "src": C4_SRC, "tgt": C4_TGT}}.get(name, {"src": srcs})
        pd.DataFrame({**base, **cols}).to_csv(OUT / f"pred_{name}_{tag}.csv", index=False)
    if with_test:
        tsrc = BEA_TEST_SRC if not QUICK_TEST else BEA_TEST_SRC[:50]; tau = test_tau     # the held-out-tuned value
        p = decode_variants(tok, m, tsrc, (0.7,), (True,), (tau,), desc=f"{tag}/bea_test")[(0.7, True, tau)]
        (OUT / "bea19_test_hyp.txt").write_text("\n".join(RETOK["bea"](x) for x in p) + "\n", encoding="utf-8")
        with zipfile.ZipFile(OUT / "bea19_test_submission_dedup.zip", "w") as z: z.write(OUT / "bea19_test_hyp.txt", "bea19_test_hyp.txt")
        r["bea_test_file"] = {"n": len(p), "tau": tau, "file": "bea19_test_submission_dedup.zip"}
    RESULTS[f"eval|{tag}"] = r; save_results()
    print(tag, json.dumps({k: (v.get("F05") if "F05" in v else v.get("GLEU")) for k, v in r.items() if isinstance(v, dict)}, default=str))


In [ ]:
# ---------------------------------------------------------------- run the jobs
NEED_H = {"final": 2.8, "nogate": 2.0} if not QUICK_TEST else {"final": 0.4, "nogate": 0.3}
for job in JOBS:
    tag = f"{job}_s2dedup"
    if RESULTS.get(f"done|{job}"): print("done already:", job); continue
    if not budget_ok(NEED_H[job], job): continue
    save = OUT / f"{job}_stage2_dedup"
    train(tag, OUT / "wi_train_dedup.parquet", OUT / "wi_holdout_dedup.parquet", S1[job], save, stage=2)
    tok, m = load_model(save)
    tau, curve = tune_tau(tok, m, HOLD_SRC, HOLD_TGT); RESULTS[f"tau|{tag}"] = {"tau": tau, "curve": curve}; save_results()
    if job == "final":
        score_all(tag, tok, m, taus=tuple(dict.fromkeys([tau, None, 0.05, 0.1, 0.2])), refines=(False, True),
                  with_test=True, test_tau=tau)
    else:
        score_all(tag, tok, m, taus=tuple(dict.fromkeys([tau, None])), refines=(True,))
    del m; gc.collect(); torch.cuda.empty_cache()
    RESULTS[f"done|{job}"] = True; save_results()


In [ ]:
# ---------------------------------------------------------------- export everything except model weights
zp = OUT / EXPORT_NAME
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.iterdir()):
        if p.is_file() and not p.name.startswith("_") and p.suffix in (".json", ".csv", ".txt", ".zip", ".png", ".tsv", ".gz") \
                and p.name != EXPORT_NAME:
            z.write(p, p.name)
print(f"wrote {zp} ({zp.stat().st_size/1e6:.1f} MB) -- download this file and send it back")
print(json.dumps({k: v for k, v in RESULTS.items() if not isinstance(v, (list,))}, indent=1, default=str)[:4000])
